In [ ]:
import os
import numpy as np
import pandas as pd
import utils as utils #커스텀 패키지
import torch
from transformers import AutoTokenizer
import ast

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

device = "cuda" if torch.cuda.is_available() else "cpu"
config = utils.load_config("./config.yaml")

#train 셋 불러오기
train_dataset = pd.read_csv(config["data"]["path"] + config["data"]["train_path"])
train_dataset["aspect_labels"] = train_dataset["aspect_labels"].apply(
    lambda x: np.array(ast.literal_eval(x), dtype=np.int64)
)
train_dataset["sentiment_labels"] = train_dataset["sentiment_labels"].apply(
    lambda x: np.array(ast.literal_eval(x), dtype=np.int64)
)

#8:2로 train/valid 분리
train_data = train_dataset.sample(frac=0.8, random_state=config['seed'])
valid_data = train_dataset.drop(train_data.index)
train_data = train_data

#### 전처리

In [2]:
#필요 시 사용
# from dataset import create_datasetV2

# train_dataset = pd.read_csv(config['data']['path'] + config['data']['train_path'])
# valid_dataset = pd.read_csv(config['data']['path'] + config['data']['val_path'])

# ASPECTS = train_dataset['Aspect'].unique()
# pd.DataFrame(ASPECTS).to_csv(config['data']['path'] + config['data']['aspect_path'], encoding='utf-8-sig')

# dataset = pd.concat([train_dataset, valid_dataset], ignore_index=True)

# train_data, test_data = create_datasetV2(dataset, ASPECTS, config['seed'])

# train_data.to_csv("./datasets/Training(5.20)_V2.csv", index=False, encoding='utf-8-sig')
# test_data.to_csv("./datasets/Test(5.20)_V2.csv", index=False, encoding='utf-8-sig')

In [3]:
#측면만 따로 뽑아서 확인
ASPECTS = pd.read_csv(config["data"]["path"] + config['data']['aspect_path'])['0'].to_list()
print(ASPECTS)

['가격', '음량/음질', '화질', '사이즈', '소음', '편의성', '디자인', '무게', '기능', '시간/속도', '조작성', '품질', '용량', '제품구성', '제조일/제조사', '색상', '내구성', '배터리', '소재']


In [4]:
#자기 모델에 맞게 수정
tokenizer = AutoTokenizer.from_pretrained(
    "monologg/koelectra-base-v3-discriminator"
)

In [5]:
from torch.utils.data import DataLoader
from dataset.make_dataset import MultiLabelABSADataset

#모델 학습용 데이터셋으로 변환
train_data_set = MultiLabelABSADataset(train_data, tokenizer, ASPECTS)

train_loader = DataLoader(
    train_data_set,
    batch_size=config["training"]["batch_size"],
    shuffle=True
)

val_data_set = MultiLabelABSADataset(valid_data, tokenizer, ASPECTS)

val_loader = DataLoader(
    val_data_set,
    batch_size=config["training"]["batch_size"],
    shuffle=False
)

#### 학습 및 검정

In [6]:
from utils import evaluate_val_multilabel
from model import ABSAModelV3, training_multi_label_model

#모델 생성
model = ABSAModelV3(num_aspect=len(ASPECTS)).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["optimizer"]["lr"]
)

epochs = config["training"]["epoch"]

#학습 재개 시
if config['train_resume'] == True:
    checkpoint = torch.load(
        config['training']["checkpoint"]["path"],
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )
    epoch = checkpoint["epoch"]
    best_epoch = checkpoint["best_epoch"]
    best_f1_score = checkpoint["best_f1_score"]
    
else:
    epoch = 0
    best_epoch = 0
    best_f1_score = 0

for epoch in range(epoch, epochs):
    print(f"epoch {epoch + 1}/{epochs}")
    avg_loss, aspect_elem_acc, aspect_subset_acc, sentiment_acc = training_multi_label_model(model, optimizer, train_loader, device)
    
    #학습 결과 출력
    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")
    print(f"Train Aspect element Acc : {aspect_elem_acc:.4f}")
    print(f"Train Aspect match Acc : {aspect_subset_acc:.4f}")
    print(f"Train Sentiment Acc : {sentiment_acc:.4f}")
    
    #검증 데이터로 과적합 및 성능 확인
    val_metrics = evaluate_val_multilabel(model, val_loader, device, threshold=0.5)
    
    print(f"Valid Aspect    F1 Score(macro): {val_metrics['aspect_macro_f1']}")
    print(f"Valid Sentiment F1 Score(macro): {val_metrics['sentiment_macro_f1']}")
    print(val_metrics)
    
    cur_f1_score = (val_metrics['aspect_macro_f1'] * 0.4) + (val_metrics['sentiment_macro_f1'] * 0.6)
    
    #sentiment f1 score 기준으로 잘 나온 모델을 저장
    if cur_f1_score > best_f1_score:
        best_epoch = epoch
        best_f1_score = cur_f1_score
        print("Best_Model_Changed")
        torch.save({
            "model_state_dict": model.state_dict(),
        }, config["model"]["save_path"])

    #checkpoint 저장
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch + 1,
        "best_epoch": best_epoch,
        "best_f1_score": best_f1_score
    }, config['training']["checkpoint"]["path"])

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 42030.51it/s]
[transformers] ElectraModel LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1/15


100%|██████████| 1970/1970 [06:32<00:00,  5.01it/s, loss=0.2554, aspect=0.1594, sentiment=0.1106, a_acc=0.9352, a_match_acc=0.078095, s_acc=0.9401]



Epoch 1 Average Loss: 0.0142
Train Aspect element Acc : 0.9352
Train Aspect match Acc : 0.0781
Train Sentiment Acc : 0.9401
Valid Aspect    F1 Score(macro): 0.0033601213519695895
Valid Sentiment F1 Score(macro): 0.7263343480340761
{'aspect_elem_acc': 0.9163781912591952, 'aspect_subset_acc': 0.0, 'aspect_micro_f1': 0.006628339533310788, 'aspect_macro_f1': 0.0033601213519695895, 'sentiment_acc': 0.9058450145464163, 'sentiment_macro_f1': 0.7263343480340761}
Best_Model_Changed
epoch 2/15


100%|██████████| 1970/1970 [06:44<00:00,  4.87it/s, loss=0.2214, aspect=0.1001, sentiment=0.1164, a_acc=0.9592, a_match_acc=0.373150, s_acc=0.9725]


...

100%|██████████| 1970/1970 [06:00<00:00,  5.46it/s, loss=0.1277, aspect=0.0865, sentiment=0.0517, a_acc=0.9707, a_match_acc=0.538651, s_acc=0.9793]



Epoch 3 Average Loss: 0.0050
Train Aspect element Acc : 0.9707
Train Aspect match Acc : 0.5387
Train Sentiment Acc : 0.9793
Valid Aspect    F1 Score(macro): 0.010907325052481475
Valid Sentiment F1 Score(macro): 0.7514777230656512
{'aspect_elem_acc': 0.889083103691726, 'aspect_subset_acc': 0.0006490696668109044, 'aspect_micro_f1': 0.01317055873562636, 'aspect_macro_f1': 0.010907325052481475, 'sentiment_acc': 0.8494225513532575, 'sentiment_macro_f1': 0.7514777230656512}
epoch 4/15


100%|██████████| 1970/1970 [06:09<00:00,  5.34it/s, loss=0.0598, aspect=0.0693, sentiment=0.0087, a_acc=0.9764, a_match_acc=0.633179, s_acc=0.9849]



Epoch 4 Average Loss: 0.0038
Train Aspect element Acc : 0.9764
Train Aspect match Acc : 0.6332
Train Sentiment Acc : 0.9849


KeyboardInterrupt: 

In [7]:
#best 모델 불러오기
save_model = torch.load(
    config['model']["save_path"],
    map_location=device
)

model.load_state_dict(
    save_model["model_state_dict"]
)

<All keys matched successfully>

In [8]:
from utils import classification_report_multilabel_absa

print("valid")
# valid
classification_report_multilabel_absa(model, val_loader, device, ASPECTS, threshold=0.5)

valid
=== Aspect Classification Report (Multi-label) ===
              precision    recall  f1-score   support

          가격       0.97      0.98      0.97      2156
       음량/음질       0.93      0.97      0.95       924
          화질       0.94      0.95      0.94       842
         사이즈       0.95      0.95      0.95      1128
          소음       0.91      0.86      0.88       263
         편의성       0.85      0.74      0.79      1167
         디자인       0.95      0.95      0.95       895
          무게       0.94      0.97      0.96       576
          기능       0.83      0.87      0.85      1921
       시간/속도       0.90      0.91      0.90       537
         조작성       0.83      0.84      0.84      1178
          품질       0.88      0.87      0.87      1058
          용량       0.94      0.92      0.93       201
        제품구성       0.92      0.84      0.87       687
     제조일/제조사       0.94      0.94      0.94       587
          색상       0.98      0.99      0.98       819
         내구성       0.94 

#### 학습 + 검증 데이터 학습, Test 셋 성능 확인

In [9]:
import gc

# 기존 모델 및 데이터 loader 삭제
del model
del optimizer
del train_loader
del val_loader

gc.collect()
torch.cuda.empty_cache()

In [15]:
#학습, 검증 데이터 생성
train_valid_dataset = pd.concat([train_data, valid_data], ignore_index=True)

train_valid_ds = MultiLabelABSADataset(train_valid_dataset, tokenizer, ASPECTS)

train_valid_loader = DataLoader(
    train_valid_ds,
    batch_size=config["training"]["batch_size"],
    shuffle=False
)

In [ ]:
#모델 생성
model = ABSAModelV3(num_aspect=len(ASPECTS)).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["optimizer"]["lr"]
)

checkpoint = torch.load(
    config['training']["checkpoint"]["path"],
    map_location=device
)

epochs = config["training"]["epoch"]

#학습 재개 시
if config['train_valid_resume'] == True:
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )
    epoch = checkpoint["epoch"]
    best_epoch = checkpoint["best_epoch"]
else:
    epoch = 0
    best_epoch = checkpoint["best_epoch"]

for epoch in range(epoch, best_epoch):
    
    print(f"epoch {epoch + 1}/{best_epoch}")
    avg_loss, aspect_elem_acc, aspect_subset_acc, sentiment_acc = training_multi_label_model(model, optimizer, train_valid_loader, device)
    #학습 결과 출력
    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")
    print(f"Train Aspect element Acc : {aspect_elem_acc:.4f}")
    print(f"Train Aspect match Acc : {aspect_subset_acc:.4f}")
    print(f"Train Sentiment Acc : {sentiment_acc:.4f}")

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch + 1,
        "best_epoch": best_epoch
    }, config['training']["checkpoint"]["path"])
    
print("Model_Saved")
torch.save({
    "model_state_dict": model.state_dict(),
}, config["model"]["train_valid_path"])

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 35797.50it/s]
[transformers] ElectraModel LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 0/9


100%|██████████| 1970/1970 [06:23<00:00,  5.13it/s, loss=0.2402, aspect=0.1752, sentiment=0.0905, a_acc=0.9361, a_match_acc=0.065801, s_acc=0.9384]



Epoch 1 Average Loss: 0.0142
Train Aspect element Acc : 0.9361
Train Aspect match Acc : 0.0658
Train Sentiment Acc : 0.9384
epoch 1/9


100%|██████████| 1970/1970 [06:26<00:00,  5.10it/s, loss=0.1987, aspect=0.1187, sentiment=0.0889, a_acc=0.9576, a_match_acc=0.344755, s_acc=0.9715]


...


Epoch 4 Average Loss: 0.0037
Train Aspect element Acc : 0.9775
Train Aspect match Acc : 0.6551
Train Sentiment Acc : 0.9854
epoch 4/9


100%|██████████| 1970/1970 [06:32<00:00,  5.02it/s, loss=0.0506, aspect=0.0524, sentiment=0.0107, a_acc=0.9816, a_match_acc=0.726534, s_acc=0.9898]



Epoch 5 Average Loss: 0.0029
Train Aspect element Acc : 0.9816
Train Aspect match Acc : 0.7265
Train Sentiment Acc : 0.9898
epoch 5/9


 40%|████      | 794/1970 [02:40<03:57,  4.94it/s, loss=0.0576, aspect=0.0558, sentiment=0.0142, a_acc=0.9836, a_match_acc=0.757321, s_acc=0.9912]


KeyboardInterrupt: 

In [ ]:
#Train_Valid 학습 모델 확인
train_valid_model = torch.load(
    config['model']["train_valid_path"],
    map_location=device
)

model.load_state_dict(
    train_valid_model["model_state_dict"]
)

In [ ]:
# test셋 성능 확인(이건 모델 선택 완료 후 사용)
# test_dataset = pd.read_csv(config["data"]["path"] + config["data"]["test_path"])
# test_dataset["aspect_labels"] = test_dataset["aspect_labels"].apply(
#     lambda x: np.array(ast.literal_eval(x), dtype=np.int64)
# )
# test_dataset["sentiment_labels"] = test_dataset["sentiment_labels"].apply(
#     lambda x: np.array(ast.literal_eval(x), dtype=np.int64)
# )

# test_data = MultiLabelABSADataset(test_dataset, tokenizer, ASPECTS)

# test_loader = DataLoader(
#     test_data,
#     batch_size=config["training"]["batch_size"],
#     shuffle=False
# )

# print("test")
# # valid
# classification_report_multilabel_absa(model, test_loader, device, ASPECTS, threshold=0.5)

#### 예시 리뷰 추론 확인

In [ ]:
#예시 리뷰 추론 확인(이건 test 셋 성능 확인 및 test 셋 학습 후 사용)
model.eval()

id2sent = {0: "negative", 1: "neutral", 2: "positive"}

def predict_multilabel_absa(sentence, model, tokenizer, aspects, device, threshold=0.5, fallback_top1=False):
    with torch.no_grad():
        enc = tokenizer(
            sentence,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=128
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        aspect_logits, sentiment_logits = model(**enc)  # [1,A], [1,A,3]

        aspect_probs = torch.sigmoid(aspect_logits)[0].cpu().numpy()      # [A]
        sentiment_ids = torch.argmax(sentiment_logits, dim=-1)[0].cpu().numpy()  # [A]

        selected_idx = np.where(aspect_probs >= threshold)[0]
        if len(selected_idx) == 0 and fallback_top1:
            selected_idx = np.array([int(np.argmax(aspect_probs))])

        results = []
        for i in selected_idx:
            results.append({
                "aspect": aspects[i],
                "aspect_prob": float(aspect_probs[i]),
                "sentiment": id2sent[int(sentiment_ids[i])]
            })
        return results

sentence = "가격은 싼데 품질이 아쉽네요"
preds = predict_multilabel_absa(
    sentence, model, tokenizer, ASPECTS, device,
    threshold=0.5,
    fallback_top1=False
)

for result in preds:
    print(result)